# Insurance Policy Data Generator

Generates a synthetic insurance book of business with 5,000 policies using Faker, NumPy, and Pandas.
Features weighted distributions, conditional logic, and correlated fields for realistic data analysis practice.

In [ ]:
from faker import Faker
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, date
import random
import re

fake = Faker('en_US')
Faker.seed(42)
random.seed(42)
np.random.seed(42)

REFERENCE_DATE = date(2026, 5, 9)

## Configuration & Distributions

In [ ]:
NUM_POLICIES = 5000

POLICY_TYPES = ['Auto', 'Home', 'Life', 'Renters', 'Umbrella', 'Motorcycle', 'Boat']
POLICY_TYPE_WEIGHTS = [0.45, 0.25, 0.12, 0.08, 0.04, 0.04, 0.02]

POLICY_STATUSES = ['Active', 'Lapsed', 'Cancelled', 'Pending Renewal', 'Expired']
STATUS_WEIGHTS = [0.70, 0.10, 0.08, 0.08, 0.04]

PAYMENT_FREQUENCIES = ['Monthly', 'Quarterly', 'Semi-Annual', 'Annual']
PAYMENT_WEIGHTS = [0.55, 0.15, 0.20, 0.10]

CHANNELS = ['Agent', 'Online', 'Phone', 'Mobile App']
CHANNEL_WEIGHTS = [0.50, 0.30, 0.12, 0.08]

MARITAL_STATUSES = ['Single', 'Married', 'Divorced', 'Widowed']
EDUCATION = ['High School', 'Some College', 'Bachelor', 'Master', 'Doctorate']
RISK_TIERS = ['Preferred', 'Standard', 'Substandard', 'High Risk']
RISK_WEIGHTS = [0.35, 0.45, 0.15, 0.05]

CLAIM_TYPES_BY_POLICY = {
    'Auto': ['Collision', 'Comprehensive', 'Liability', 'Uninsured Motorist'],
    'Home': ['Wind/Hail', 'Water Damage', 'Fire', 'Theft', 'Liability'],
    'Life': ['Death Benefit'],
    'Renters': ['Theft', 'Fire', 'Water Damage'],
    'Umbrella': ['Liability'],
    'Motorcycle': ['Collision', 'Comprehensive', 'Liability'],
    'Boat': ['Collision', 'Storm Damage', 'Theft']
}

for name, w in [('POLICY_TYPE_WEIGHTS', POLICY_TYPE_WEIGHTS), ('STATUS_WEIGHTS', STATUS_WEIGHTS),
                 ('PAYMENT_WEIGHTS', PAYMENT_WEIGHTS), ('CHANNEL_WEIGHTS', CHANNEL_WEIGHTS),
                 ('RISK_WEIGHTS', RISK_WEIGHTS)]:
    assert abs(sum(w) - 1.0) < 0.001, f"{name} must sum to 1.0"

## Helper Functions

In [ ]:
def generate_premium(policy_type, risk_tier, coverage_amount, safe_driver=False):
    base_rates = {
        'Auto': 0.04, 'Home': 0.005, 'Life': 0.008, 'Renters': 0.015,
        'Umbrella': 0.002, 'Motorcycle': 0.05, 'Boat': 0.015
    }
    risk_multipliers = {
        'Preferred': 0.85, 'Standard': 1.0, 'Substandard': 1.35, 'High Risk': 1.75
    }
    base = coverage_amount * base_rates[policy_type] * risk_multipliers[risk_tier]
    if safe_driver:
        base *= 0.92
    return round(base * np.random.uniform(0.85, 1.15), 2)

def generate_coverage(policy_type):
    ranges = {
        'Auto': (25000, 300000),
        'Home': (150000, 800000),
        'Life': (50000, 2000000),
        'Renters': (15000, 100000),
        'Umbrella': (1000000, 5000000),
        'Motorcycle': (15000, 50000),
        'Boat': (20000, 250000)
    }
    low, high = ranges[policy_type]
    raw = np.random.uniform(low, high)
    rounded = round(raw, -3)
    return max(rounded, low)

def _clamped_date_between(start_earliest, end_latest):
    if start_earliest >= end_latest:
        return start_earliest + (end_latest - start_earliest) // 2
    return fake.date_between(start_date=start_earliest, end_date=end_latest)

def generate_policy_end_date(policy_start, policy_status):
    if policy_status == 'Active':
        return policy_start + timedelta(days=365 * random.choice([1, 1, 1, 2, 3]))
    elif policy_status == 'Pending Renewal':
        return REFERENCE_DATE + timedelta(days=random.randint(1, 90))
    elif policy_status == 'Lapsed':
        return _clamped_date_between(policy_start + timedelta(days=180), REFERENCE_DATE - timedelta(days=180))
    elif policy_status == 'Cancelled':
        return _clamped_date_between(policy_start + timedelta(days=30), REFERENCE_DATE - timedelta(days=30))
    else:
        return _clamped_date_between(REFERENCE_DATE - timedelta(days=180), REFERENCE_DATE)

def normalize_phone(phone):
    digits = re.sub(r'\D', '', phone)
    if len(digits) == 10:
        return f'({digits[:3]}) {digits[3:6]}-{digits[6:]}'
    if len(digits) == 11 and digits[0] == '1':
        digits = digits[1:]
        return f'({digits[:3]}) {digits[3:6]}-{digits[6:]}'
    return phone

def build_demographics():
    dob = fake.date_of_birth(minimum_age=22, maximum_age=85)
    age = (REFERENCE_DATE - dob).days // 365
    return {
        'first_name': fake.first_name(),
        'last_name': fake.last_name(),
        'date_of_birth': dob,
        'age': age,
        'gender': np.random.choice(['M', 'F'], p=[0.49, 0.51]),
        'marital_status': random.choice(MARITAL_STATUSES),
        'education': random.choice(EDUCATION),
        'occupation': fake.job(),
        'annual_income': round(np.random.lognormal(11, 0.5), -2),
        'credit_score': int(np.clip(np.random.normal(720, 80), 300, 850)),
    }

def build_contact():
    state = fake.state_abbr()
    return {
        'email': fake.email(),
        'phone': normalize_phone(fake.phone_number()),
        'address': fake.street_address(),
        'city': fake.city(),
        'state': state,
        'zip_code': fake.zipcode(),
    }

def build_policy_terms(i, policy_type, risk_tier, coverage, annual_premium, policy_start, policy_end):
    return {
        'policy_id': f'POL-{1000000 + i}',
        'customer_id': f'CUST-{500000 + i}',
        'policy_type': policy_type,
        'policy_status': np.random.choice(POLICY_STATUSES, p=STATUS_WEIGHTS),
        'risk_tier': risk_tier,
        'coverage_amount': coverage,
        'deductible': random.choice([250, 500, 1000, 2500, 5000]),
        'annual_premium': annual_premium,
        'monthly_premium': round(annual_premium / 12, 2),
        'payment_frequency': np.random.choice(PAYMENT_FREQUENCIES, p=PAYMENT_WEIGHTS),
        'policy_start_date': policy_start,
        'policy_end_date': policy_end,
        'tenure_years': round((REFERENCE_DATE - policy_start).days / 365, 1),
        'acquisition_channel': np.random.choice(CHANNELS, p=CHANNEL_WEIGHTS),
        'agent_id': f'AGT-{random.randint(1000, 1500)}',
        'has_multi_policy_discount': np.random.choice([True, False], p=[0.4, 0.6]),
    }

def build_claims(policy_type, policy_start, num_claims):
    if num_claims == 0:
        return {
            'num_claims': 0,
            'total_claims_paid': 0.0,
            'last_claim_date': pd.NA,
            'primary_claim_type': pd.NA,
        }
    return {
        'num_claims': num_claims,
        'total_claims_paid': round(sum(np.random.uniform(500, 50000) for _ in range(num_claims)), 2),
        'last_claim_date': fake.date_between(start_date=policy_start, end_date=REFERENCE_DATE),
        'primary_claim_type': random.choice(CLAIM_TYPES_BY_POLICY[policy_type]),
    }

## Generate Dataset

In [ ]:
records = []

for i in range(NUM_POLICIES):
    policy_type = np.random.choice(POLICY_TYPES, p=POLICY_TYPE_WEIGHTS)
    risk_tier = np.random.choice(RISK_TIERS, p=RISK_WEIGHTS)
    coverage = generate_coverage(policy_type)

    safe_driver = policy_type in ['Auto', 'Motorcycle'] and np.random.choice([True, False], p=[0.55, 0.45])
    annual_premium = generate_premium(policy_type, risk_tier, coverage, safe_driver=safe_driver)

    policy_start = fake.date_between(start_date='-5y', end_date=REFERENCE_DATE)
    policy_status = np.random.choice(POLICY_STATUSES, p=STATUS_WEIGHTS)
    policy_end = generate_policy_end_date(policy_start, policy_status)

    num_claims = np.random.choice([0, 1, 2, 3, 4, 5], p=[0.65, 0.20, 0.08, 0.04, 0.02, 0.01])

    record = {}
    record.update(build_demographics())
    record.update(build_contact())
    record.update(build_policy_terms(i, policy_type, risk_tier, coverage, annual_premium, policy_start, policy_end))
    record.update(build_claims(policy_type, policy_start, num_claims))
    record['has_safe_driver_discount'] = safe_driver
    record['nps_score'] = random.randint(0, 10)
    record['customer_lifetime_value'] = round(annual_premium * np.random.uniform(3, 15), 2)

    records.append(record)

df = pd.DataFrame(records)

## Optimize Dtypes

In [ ]:
CATEGORICAL_COLS = [
    'policy_type', 'policy_status', 'risk_tier', 'gender', 'marital_status',
    'education', 'payment_frequency', 'acquisition_channel', 'state',
    'primary_claim_type'
]
for col in CATEGORICAL_COLS:
    if col in df.columns:
        df[col] = df[col].astype('category')

## Sanity Check

In [ ]:
print(f"Generated {len(df):,} policies")
print(f"\nColumns ({len(df.columns)}): {list(df.columns)}")
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")
print(f"\nPolicy type distribution:")
print(df['policy_type'].value_counts())
print(f"\nNumeric summary:")
print(df[['annual_premium', 'age', 'credit_score', 'annual_income', 'num_claims']].describe())
print(f"\nFirst few rows:")
df.head()

## Export Data

In [ ]:
df.to_csv('../data/raw/insurance_policies.csv', index=False)
print("Saved to ../data/raw/insurance_policies.csv")